# Mission Control — laboratório eBPF (libbpf) + Docker

No **host** basta ter Docker em Linux com BTF (`/sys/kernel/btf/vmlinux`). **Não** é necessário instalar `clang`, `llvm` nem `bpftool` no host: a compilação e o `./loader` rodam dentro do container **`ebpf_loader`** (privilegiado, rede e PID do host, volumes para BTF, bpf e debug).

Abra este notebook com o diretório de trabalho na pasta **`mission-control-lab`** (onde estão `Makefile`, `docker-compose.yml`, `Dockerfile.loader` e os `.c`).

### 1. Subir apps em deadlock + container do loader

`docker compose up -d` sobe `mc_app_a`, `mc_app_b` na rede `mission_control_net` e o `ebpf_loader` em segundo plano (`tail -f /dev/null`).

In [ ]:
!docker compose up -d --build

### 2. Compilar eBPF e o loader **dentro** do `ebpf_loader`

O diretório do projeto está montado em `/opt/mission`; os artefatos (`vmlinux.h`, `.o`, `.skel.h`, binário `loader`) aparecem também no host.

In [ ]:
!docker exec ebpf_loader sh -c 'cd /opt/mission && make clean 2>/dev/null; make all'

### 3. Estado dos containers e da rede Docker

In [ ]:
!docker ps -a --filter "name=mc_app" ; docker ps -a --filter "name=ebpf_loader" ; echo "---" && docker network inspect mission_control_net --format '{{json .}}' | head -c 2000

### 4. Rodar o loader no container (TC na `br-…` + ringbuf)

O `ebpf_loader` usa `network_mode: host`, então vê a interface `br-` da rede `mission_control_net`. Esta célula **bloqueia** até você interromper a execução do notebook (Stop) ou o processo terminar; em outro terminal pode acompanhar `docker logs mc_app_a` / `mc_app_b`.

**Sem `sudo` no host** — o processo já roda como root dentro do container privilegiado.

In [ ]:
!BR=$(docker network inspect mission_control_net -f '{{.Id}}' | head -c 12) && echo "Bridge: br-$BR" && docker exec ebpf_loader sh -c "cd /opt/mission && ./loader br-$BR"

### 5. Logs dos apps (RST / saída do `recv`)

Após ~5 s sem payload TCP nas conexões ativas, o kernel deve registrar o alerta e o RST; os processos saem do `recv()`.

In [ ]:
!docker logs mc_app_a 2>&1 | tail -n 50
!echo '---'
!docker logs mc_app_b 2>&1 | tail -n 50

### Encerrar (opcional)

In [ ]:
!docker compose down